# Train mini-llm on Colab

Prepares the training data and trains the model on a Colab GPU while this notebook file stays on your own machine.

**Before running:** choose `Select Kernel` > `Colab` > `New Colab Server` and pick a machine with a GPU, for example a T4.

Run the cells from top to bottom once per Colab session. Training runs in the background and writes to a log file, so the cells under *Watch progress* can be re-run at any time. After a kernel restart, re-run the two cells under *Settings and helpers* first; they start nothing.

Everything on the Colab machine is deleted when the session ends. The training data, the checkpoints and the log are therefore kept on Google Drive, and a new session carries on from the last step that was saved there.

## Settings and helpers

These two cells only define names. Run them first, in every session and after every kernel restart.

**Space on Drive.** A checkpoint of the full model is 1.95 GB. The run keeps two rolling checkpoints, up to `KEEP_CHECKPOINTS` end-of-epoch checkpoints, and at the end the 0.65 GB of final weights. One epoch with the settings below needs about 6.5 GB and five epochs about 14.3 GB; with `KEEP_CHECKPOINTS = 2` five epochs need about 8.5 GB. The training data takes another 4 GB. A checkpoint the script deletes goes to Drive's Trash and keeps using space until you empty the Trash.

In [ ]:
REPO = "crrankyy/mini-llm"
BRANCH = "main"

# on the Colab machine: temporary
REPO_DIR = "/content/mini-llm"

# on Google Drive: kept. The folder and the log are named after the context length,
# because a model with another context length cannot resume from these checkpoints
DRIVE_DIR = "/content/drive/MyDrive/mini-llm"
CHECKPOINT_DIR = f"{DRIVE_DIR}/checkpoints-1024"
LOG_FILE = f"{DRIVE_DIR}/train-1024.log"

# the training data: this many tokens, this share of them from FineWeb-Edu and the rest
# from the general-web FineWeb. Kept on Drive too, and built there only if it is missing
DATA_DIR = f"{DRIVE_DIR}/data/fineweb-mix"
TRAIN_TOKENS = 1_000_000_000
EDU_SHARE = 0.5
# passes over the training data in total, across all sessions
EPOCHS = 1

# windows of 1,024 tokens in one batch. If the GPU runs out of memory, halve this and
# double ACCUMULATE and SAVE_EVERY: an update and a save then cover as much text as before
BATCH_SIZE = 4
# one weight update is made from the gradients of this many batches
ACCUMULATE = 16
# the learning rate rises to LEARNING_RATE during the first WARMUP_SHARE of all weight
# updates, then falls along a cosine to a tenth of it at the end of the last epoch
LEARNING_RATE = 6e-4
WARMUP_SHARE = 0.03

# training steps between saves of the rolling checkpoint: the most a session can lose
SAVE_EVERY = 2000
# end-of-epoch checkpoints to keep; older ones are deleted
KEEP_CHECKPOINTS = 10

In [ ]:
import getpass
import os
import re
import signal
import subprocess
import sys

import matplotlib.pyplot as plt

# the line train_model_v1 prints at every evaluation
EVALUATION = re.compile(
    r"Ep (\d+) \(Step (\d+)\) \[(.+?)\]: Train loss ([\d.]+|nan|inf), Val loss ([\d.]+|nan|inf)"
)

def clone_repo(repo, branch, target, token):
    # the first -c clears any credential helper already configured; the second answers
    # git's password prompt from the environment, so the token never appears in a
    # command line or in git's config
    helper = "!f() { echo username=x-access-token; echo password=$GITHUB_TOKEN; }; f"
    return subprocess.run(
        ["git", "-c", "credential.helper=", "-c", f"credential.helper={helper}",
         "clone", "--branch", branch, f"https://github.com/{repo}.git", target],
        env={**os.environ, "GITHUB_TOKEN": token, "GIT_TERMINAL_PROMPT": "0"},
        capture_output=True, text=True,
    )

def training_pid():
    # -o picks the oldest match: the main process, not one of its data-loading workers
    result = subprocess.run(
        ["pgrep", "-o", "-f", "scripts/train_wikitext.py"], capture_output=True, text=True
    )
    return int(result.stdout) if result.stdout.strip() else None

def start_training():
    with open(LOG_FILE, "a") as log:
        process = subprocess.Popen(
            # -u: unbuffered, so each line reaches the log as soon as it is printed
            [sys.executable, "-u", "scripts/train_wikitext.py", "--checkpoint-dir", CHECKPOINT_DIR,
             "--data-dir", DATA_DIR, "--epochs", str(EPOCHS),
             "--save-every", str(SAVE_EVERY), "--keep-checkpoints", str(KEEP_CHECKPOINTS),
             "--batch-size", str(BATCH_SIZE), "--accumulate", str(ACCUMULATE),
             "--lr", str(LEARNING_RATE), "--warmup-share", str(WARMUP_SHARE)],
            cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT,
            # its own session, so interrupting or restarting the kernel does not stop it
            start_new_session=True,
        )
    return process.pid

def read_evaluations(log_file):
    if not os.path.exists(log_file):
        return []

    with open(log_file) as f:
        return [
            {"epoch": int(m[1]), "step": int(m[2]), "time": m[3], "train": float(m[4]), "val": float(m[5])}
            for m in EVALUATION.finditer(f.read())
        ]

def plot_evaluations(evaluations):
    plt.figure(figsize=(8, 4))
    plt.plot([e["train"] for e in evaluations], label="train")
    plt.plot([e["val"] for e in evaluations], label="validation")
    # the step counter restarts whenever training resumes, so the axis counts evaluations
    plt.xlabel("evaluation (about one every 500 steps)")
    plt.ylabel("loss")
    plt.legend()
    plt.show()

## 1. Check the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Pick a Colab server with a GPU in the kernel picker."
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

## 2. Mount Google Drive

A browser window asks for permission the first time.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# without the mount, the Drive paths above would be ordinary folders on the Colab machine
# and everything written to them would be deleted with it
assert os.path.ismount("/content/drive"), "Google Drive is not mounted."
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 3. Get the code

The repository is private, so cloning needs a GitHub token that can read it. A fine-grained personal access token limited to this one repository, with read access to *Contents*, is enough.

The token is handed to git through the environment. It is not written to the Colab machine's disk or to this notebook.

In [ ]:
if os.path.isdir(REPO_DIR):
    print(f"{REPO_DIR} already exists; leaving it as it is.")
else:
    result = clone_repo(REPO, BRANCH, REPO_DIR, getpass.getpass("GitHub token: "))
    print(result.stderr.strip())
    assert result.returncode == 0, "git clone failed"

## 4. Install

Installs the `mini_llm` package from the clone, plus `datasets` and `tqdm` for the data step. Colab's own PyTorch is kept.

In [ ]:
%pip install -q -e "{REPO_DIR}[data]"

## 5. Prepare the data

`DATA_DIR` on Google Drive holds the prepared data: `TRAIN_TOKENS` tokens, 4 bytes each, drawn evenly from every file of FineWeb-Edu and of FineWeb. If the folder already holds data prepared with the settings above, this step does nothing. If not, it downloads and tokenizes the data into that folder, always from the same pinned version of both datasets.

Training does not read the files from Drive. When it starts, it copies them to the Colab machine's own disk, once per session.

Keep `TRAIN_TOKENS` and `EDU_SHARE` the same for as long as you continue one training run. A run that resumes expects the data it started with; for different data, use a new checkpoint folder.

In [ ]:
# the script does nothing if the folder already holds data prepared with these settings.
# It is run with the kernel's own interpreter, which is where the package was just installed
!cd "{REPO_DIR}" && "{sys.executable}" scripts/prepare_fineweb.py --tokens {TRAIN_TOKENS} --edu-share {EDU_SHARE} --output-dir "{DATA_DIR}"

## 6. Start training

Starts `scripts/train_wikitext.py` as a background process. If the checkpoint folder on Drive holds checkpoints from an earlier session, the script resumes from the newest one, at the step it was saved at.

Keep `EPOCHS`, `BATCH_SIZE`, `ACCUMULATE`, `LEARNING_RATE` and `WARMUP_SHARE` the same for as long as you continue one training run. The learning rate at every update is worked out from them, so a change moves the rest of the run onto a different curve. That includes raising `EPOCHS` after the last epoch has finished: the rate would jump back up. A start with a different `BATCH_SIZE` is refused, because the saved position is counted in batches.

Running this cell again while training is still running does nothing.

In [ ]:
pid = training_pid()
if pid is None:
    print("Started training, pid", start_training())
else:
    print("Training is already running, pid", pid)

## 7. Watch progress

Re-run these two cells whenever you want an update. The first plots the losses, the second shows the end of the log, which includes the sample text printed after each epoch and any error.

In [ ]:
evaluations = read_evaluations(LOG_FILE)
if evaluations:
    latest = evaluations[-1]
    print(
        f"{len(evaluations)} evaluations. Latest: epoch {latest['epoch']}, step {latest['step']}, "
        f"{latest['time']}, train {latest['train']:.3f}, validation {latest['val']:.3f}"
    )
    plot_evaluations(evaluations)
else:
    print("No evaluations in the log yet.")

In [ ]:
pid = training_pid()
print(f"Training is running, pid {pid}." if pid else "Training is not running.")

!tail -n 20 "{LOG_FILE}"
!ls -lh "{CHECKPOINT_DIR}"

## 8. Stop training

Only needed to stop early. The first cell sends the same interrupt as Ctrl-C in a terminal. The script then saves the step it had reached to a rolling checkpoint, which takes a while for a file of this size, and the next session resumes from exactly there.

Before disconnecting, wait until the status cell above says training is not running, then run the second cell. It returns once everything written to Drive has been uploaded.

In [ ]:
pid = training_pid()
if pid is None:
    print("Training is not running.")
else:
    # the whole process group, as Ctrl-C in a terminal would do
    os.killpg(os.getpgid(pid), signal.SIGINT)
    print("Sent the interrupt to pid", pid)

In [ ]:
from google.colab import drive

# unmounting under a running script would cut off its log and its checkpoints
assert training_pid() is None, "Training is still running."
drive.flush_and_unmount()

## Sampling from a checkpoint on your own machine

Download a file from `mini-llm/checkpoints-1024/` on Google Drive into the `checkpoints/` folder of your local clone, then run from the repo root:

```sh
uv run python scripts/generate.py --weights checkpoints/gpt_wikitext_checkpoint_ep1.pth
```

Without `--weights`, the script loads `checkpoints/gpt_wikitext_124M.pth` if it is there and otherwise the checkpoint a training run would resume from: the newer rolling checkpoint, or failing that the latest end-of-epoch one.